In [1]:
import pandas as pd
import numpy as np
import json
import re
import unicodedata
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier
import preprocessor as p
from Rumors_Classifier.utils import parse_propagation_file
import emoji
from camel_tools.utils.normalize import (
    normalize_alef_ar,
    normalize_alef_maksura_ar,
    normalize_teh_marbuta_ar
)


## helper functions

In [2]:
def extract_features(text):
    # Use preprocessor to get structured entities
    parsed = p.parse(text)

    # 1. Basic Length Features
    char_len = len(text)  # characters per tweet
    word_len = len(text.split())  # words per tweet

    # 2. Extracted Entities
    hashtags = len(parsed.hashtags) if parsed.hashtags else 0
    mentions = len(parsed.mentions) if parsed.mentions else 0
    urls = len(parsed.urls) if parsed.urls else 0
    emojis = 0
    for character in text:
        if emoji.is_emoji(character):
            emojis += 1

    # 3. Exclamation Count
    exclamations = text.count('!')

    return pd.Series([char_len, word_len, hashtags, mentions, urls, emojis, exclamations])

In [3]:
class ArabicTextPreprocessor(BaseEstimator, TransformerMixin):
    """
    Arabic text normalization for NLP classification tasks.
    Suitable for rumor/fake-news detection.
    """

    def __init__(
        self,
        remove_urls=True,
        remove_mentions=False,
        remove_emojis=False,
        remove_numbers=True,
        remove_punctuation=True,
        reduce_repeated_chars=True,
    ):
        self.remove_urls = remove_urls
        self.remove_mentions = remove_mentions
        self.remove_emojis = remove_emojis
        self.remove_numbers = remove_numbers
        self.remove_punctuation = remove_punctuation
        self.reduce_repeated_chars = reduce_repeated_chars

        # Arabic diacritics
        self.diacritics_pattern = re.compile(
            r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]'
        )

        # Tatweel
        self.tatweel_pattern = re.compile(r'ـ')

        # URLs
        self.url_pattern = re.compile(
            r'https?://\S+|www\.\S+'
        )

        # Mentions
        self.mention_pattern = re.compile(r'@\w+')

        # Hashtag symbol only
        self.hashtag_pattern = re.compile(r'#')

        # Repeated characters
        self.repeat_pattern = re.compile(r'(.)\1{2,}')

        # Numbers
        self.number_pattern = re.compile(r'\d+')

        # Punctuation
        self.punct_pattern = re.compile(
            r'[^\w\s\u0600-\u06FF]'
        )

        # Emoji pattern
        self.emoji_pattern = re.compile(
            "["
            "\U0001F600-\U0001F64F"
            "\U0001F300-\U0001F5FF"
            "\U0001F680-\U0001F6FF"
            "\U0001F1E0-\U0001F1FF"
            "\U00002700-\U000027BF"
            "\U000024C2-\U0001F251"
            "]+",
            flags=re.UNICODE,
        )

    def fit(self, X, y=None):
        return self

    def _normalize_text(self, text):

        if not isinstance(text, str):
            text = str(text)

        # Unicode normalization
        text = unicodedata.normalize("NFKC", text)

        # Remove URLs
        if self.remove_urls:
            text = self.url_pattern.sub(" ", text)

        # Remove mentions
        if self.remove_mentions:
            text = self.mention_pattern.sub(" ", text)

        # Keep hashtag word, remove #
        text = self.hashtag_pattern.sub("", text)

        # Remove diacritics
        text = self.diacritics_pattern.sub("", text)

        # Remove tatweel
        text = self.tatweel_pattern.sub("", text)

        # CAMeL normalization
        text = normalize_alef_ar(text)
        text = normalize_alef_maksura_ar(text)
        text = normalize_teh_marbuta_ar(text)

        # Reduce elongated words
        if self.reduce_repeated_chars:
            text = self.repeat_pattern.sub(r"\1\1", text)

        # Remove emojis
        if self.remove_emojis:
            text = self.emoji_pattern.sub("", text)

        # Remove numbers
        if self.remove_numbers:
            text = self.number_pattern.sub("", text)

        # Remove punctuation
        if self.remove_punctuation:
            text = self.punct_pattern.sub(" ", text)

        # Remove extra whitespace
        text = re.sub(r"\s+", " ", text).strip()

        return text

    def transform(self, X):
        return [self._normalize_text(text) for text in X]

After testing the effect of each of the text normalization steps and their cobination: remove_numbers, remove_mentions and remove_urls had notable positive effects( you can check logs/baseline_metrics_* and logs/norm_grid_results for reference).

Combinations are evaluated later on.

In [4]:
def prepare_data(tweets_path, replies_path, retweets_path):
    # Load
    tweets = pd.read_csv(tweets_path, sep='\t')
    tweets['tweetID'] = tweets['tweetID'].astype(str)

    tweets['text_dup_count'] = tweets.groupby('tweetText')['tweetText'].transform('count')
    # Deduplicate
    tweets = tweets.drop_duplicates(subset=['tweetText'], keep='first')

    text_preprocessor = ArabicTextPreprocessor()
    tweets["normalized_text"] = text_preprocessor.transform(X=tweets["tweetText"])

    # Text features
    p.set_options(p.OPT.URL, p.OPT.MENTION, p.OPT.HASHTAG, p.OPT.EMOJI)
    tweets[['char_len', 'word_len', 'num_hashtags', 'num_mentions', 'num_urls', 'num_emojis', 'num_exclamations']] = tweets['normalized_text'].apply(extract_features)


    # Propagation
    reply_counts = parse_propagation_file(replies_path)
    retweet_counts = parse_propagation_file(retweets_path)
    tweets['num_replies'] = tweets['tweetID'].map(reply_counts).fillna(0).astype(int)
    tweets['num_retweets'] = tweets['tweetID'].map(retweet_counts).fillna(0).astype(int)

    # Average word length
    tweets['avg_word_length'] = tweets['char_len'] / tweets['word_len'].replace(0, np.nan)

    # Drop columns
    tweets = tweets.drop(['word_len'], axis=1, errors='ignore')

    return tweets

In [5]:
tweets_path ="../../ArCOV19-Rumors/tweet_verification/Tweets.txt"
replies_path='../../ArCOV19-Rumors/tweet_verification/propagation_networks/replies'
retweets_path='../../ArCOV19-Rumors/tweet_verification/propagation_networks/retweets'

tweets_df = prepare_data(tweets_path, replies_path, retweets_path)

In [6]:
from sklearn.model_selection import train_test_split

# Set random state for reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2

# First split: separate test set
X_temp, X_test, y_temp, y_test = train_test_split(
    tweets_df, tweets_df['label'], test_size=TEST_SIZE, stratify=tweets_df['label'], random_state=RANDOM_STATE
)

# Then split the temporary set into train and validation (val = 0.25 of temp equals 20% of total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=VAL_SIZE/(1-TEST_SIZE), stratify=y_temp, random_state=RANDOM_STATE
)

print(f"Train size: {len(X_train)}")
print(f"Validation size: {len(X_val)}")
print(f"Test size: {len(X_test)}")
print(f"Train label ratio: {y_train.mean():.3f}")
print(f"Val label ratio: {y_val.mean():.3f}")
print(f"Test label ratio: {y_test.mean():.3f}")

Train size: 2078
Validation size: 693
Test size: 693
Train label ratio: 0.507
Val label ratio: 0.506
Test label ratio: 0.506


In [7]:
handcrafted_cols = [
    'char_len', 'avg_word_length', 'num_hashtags', 'num_mentions',
    'num_urls', 'num_emojis', 'num_exclamations', 'num_replies', 'num_retweets'
]

In [8]:
def evaluate_handcrafted_features(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    handcrafted_cols,
    tfidf_max_features=5000
):
    """
    Performs ablation: removes each handcrafted feature one by one,
    retrains XGBoost (with early stopping) and reports F1 on test set.
    Also reports baseline (using all features).
    """

    results = []
    # baseline: all features
    all_features = handcrafted_cols

    train_text = X_train['normalized_text']
    val_text = X_val['normalized_text']
    test_text = X_test['normalized_text']

    def build_and_evaluate(feature_subset, name):
        # Build ColumnTransformer with current subset
        preprocessor = ColumnTransformer([
            ('handcrafted', StandardScaler(), feature_subset),
            ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=tfidf_max_features, sublinear_tf=True), 'text')
        ])

        # Prepare dataframes with preprocessed text
        train_df = X_train[feature_subset].copy()
        train_df['text'] = train_text
        val_df = X_val[feature_subset].copy()
        val_df['text'] = val_text

        X_train_comb = preprocessor.fit_transform(train_df)
        X_val_comb = preprocessor.transform(val_df)

        # Compute scale_pos_weight
        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        scale_pos_weight = neg / pos if pos > 0 else 1.0

        model = XGBClassifier(
            objective='binary:logistic',
            scale_pos_weight=scale_pos_weight,
            n_estimators=2000,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            eval_metric='logloss',
            random_state=42,
            early_stopping_rounds=20
        )
        model.fit(X_train_comb, y_train, eval_set=[(X_val_comb, y_val)], verbose=False)

        # Test evaluation
        test_df = X_test[feature_subset].copy()
        test_df['text'] = test_text
        X_test_comb = preprocessor.transform(test_df)
        y_pred = model.predict(X_test_comb)
        f1 = f1_score(y_test, y_pred)
        return f1

    # Baseline
    baseline_f1 = build_and_evaluate(all_features, "All features")
    results.append({"Dropped_Feature": "None (Baseline)", "Test_F1": baseline_f1})

    # Ablate each feature
    for feature in handcrafted_cols:
        subset = [f for f in handcrafted_cols if f != feature]
        f1_score_val = build_and_evaluate(subset, f"Without {feature}")
        results.append({"Dropped_Feature": feature, "Test_F1": f1_score_val})

    results_df = pd.DataFrame(results)
    # Add a column showing drop from baseline
    results_df["Drop_F1"] = baseline_f1 - results_df["Test_F1"]
    results_df = results_df.sort_values("Drop_F1", ascending=False)
    return results_df

In [9]:
handcrafted_results = evaluate_handcrafted_features(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    handcrafted_cols
)
print(handcrafted_results)

    Dropped_Feature   Test_F1   Drop_F1
1          char_len  0.877143  0.017518
9      num_retweets  0.884120  0.010541
8       num_replies  0.886657  0.008004
3      num_hashtags  0.887304  0.007357
4      num_mentions  0.887304  0.007357
5          num_urls  0.887304  0.007357
6        num_emojis  0.887304  0.007357
7  num_exclamations  0.887304  0.007357
2   avg_word_length  0.893010  0.001651
0   None (Baseline)  0.894661  0.000000


In [10]:
handcrafted_cols = [
    'char_len', 'avg_word_length', 'num_hashtags', 'num_mentions',
    'num_urls', 'num_emojis', 'num_exclamations', 'num_replies', 'num_retweets'
]

# Define column transformer
preprocessor = ColumnTransformer([
    ('handcrafted', StandardScaler(), handcrafted_cols),
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True), 'normalized_text')
])

In [11]:
# scaling weights for imbalance
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

xgb_model = XGBClassifier(
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight,
    n_estimators=2000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    early_stopping_rounds=20,
)

X_train_transformed = preprocessor.fit_transform(X_train)
X_val_transformed = preprocessor.transform(X_val)

# Then train XGBoost with early stopping
eval_set = [(X_val_transformed, y_val)]
xgb_model.fit(
    X_train_transformed, y_train,
    eval_set=eval_set,
    verbose=True
)

# Now wrap the trained model and preprocessor together for saving
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

scale_pos_weight = 0.97
[0]	validation_0-logloss:0.67927
[1]	validation_0-logloss:0.66871
[2]	validation_0-logloss:0.65915
[3]	validation_0-logloss:0.64973
[4]	validation_0-logloss:0.64006
[5]	validation_0-logloss:0.63136
[6]	validation_0-logloss:0.62346
[7]	validation_0-logloss:0.61644
[8]	validation_0-logloss:0.60803
[9]	validation_0-logloss:0.60097
[10]	validation_0-logloss:0.59333
[11]	validation_0-logloss:0.58807
[12]	validation_0-logloss:0.58140
[13]	validation_0-logloss:0.57599
[14]	validation_0-logloss:0.57051
[15]	validation_0-logloss:0.56599
[16]	validation_0-logloss:0.56129
[17]	validation_0-logloss:0.55585
[18]	validation_0-logloss:0.55122
[19]	validation_0-logloss:0.54644
[20]	validation_0-logloss:0.54202
[21]	validation_0-logloss:0.53709
[22]	validation_0-logloss:0.53305
[23]	validation_0-logloss:0.52885
[24]	validation_0-logloss:0.52524
[25]	validation_0-logloss:0.52156
[26]	validation_0-logloss:0.51672
[27]	validation_0-logloss:0.51237
[28]	validation_0-logloss:0.50842


In [12]:
X_test_transformed = preprocessor.transform(X_test)
y_pred_proba = xgb_model.predict_proba(X_test_transformed)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)


accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)
cm = confusion_matrix(y_test, y_pred)

print("\n=== Baseline XGBoost (Combined Features) ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-macro:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print(f"PR-AUC:    {pr_auc:.4f}")
print(f"Confusion Matrix:\n{cm}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['False', 'True']))

# Save metrics to JSON
metrics = {
    'accuracy': accuracy,
    'f1_macro': f1,
    'roc_auc': roc_auc,
    'pr_auc': pr_auc,
    'confusion_matrix': cm.tolist(),
    'train_size': len(X_train),
    'val_size': len(X_val),
    'test_size': len(X_test),
    'scale_pos_weight': scale_pos_weight,
    'early_stopping_rounds': xgb_model.best_iteration,
    'best_score': xgb_model.best_score
}

with open('logs/baseline_metrics_finalized2', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to logs/baseline_metrics_finalized2.json")


=== Baseline XGBoost (Combined Features) ===
Accuracy:  0.8947
F1-macro:  0.8947
ROC-AUC:   0.9502
PR-AUC:    0.9561
Confusion Matrix:
[[310  32]
 [ 41 310]]

Classification Report:
              precision    recall  f1-score   support

       False       0.88      0.91      0.89       342
        True       0.91      0.88      0.89       351

    accuracy                           0.89       693
   macro avg       0.89      0.89      0.89       693
weighted avg       0.89      0.89      0.89       693

Metrics saved to logs/baseline_metrics_finalized2.json


In [13]:
import joblib

# Save the full pipeline (including preprocessor and trained classifier)
joblib.dump(final_pipeline, 'models/xgb_combined_pipeline.pkl')
print("Model saved to models/xgb_combined_pipeline.pkl")

# Also save just the preprocessor separately (optional, for inference flexibility)
joblib.dump(preprocessor, 'models/preprocessor.pkl')

Model saved to models/xgb_combined_pipeline.pkl


['models/preprocessor.pkl']

In [14]:
from itertools import product
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import f1_score
import pandas as pd

def evaluate_preprocessing_configs_on_test(
    X_train, X_val, X_test,
    y_train, y_val, y_test,
    text_col='normalized_text',
    handcrafted_cols=None
):
    """
    Test preprocessing combinations using the full feature set.
    Trains on X_train, validates on X_val (early stopping), evaluates on X_test.
    Returns DataFrame with configs and test F1 scores.
    """
    if handcrafted_cols is None:
        handcrafted_cols = [
            'char_len', 'avg_word_length', 'num_hashtags', 'num_mentions',
            'num_urls', 'num_emojis', 'num_exclamations', 'num_replies', 'num_retweets'
        ]

    params = ["remove_urls", "remove_mentions", "remove_numbers","remove_emojis", "remove_punctuation", "reduce_repeated_chars"]
    results = []
    RANDOM_STATE = 42

    for values in product([False, True], repeat=len(params)):
        config = dict(zip(params, values))

        # 1. Preprocess raw text for train, val, test
        train_text = X_train[text_col]
        val_text = X_val[text_col]
        test_text = X_test[text_col]

        # 2. Build column transformer
        transformer = ColumnTransformer([
            ('handcrafted', StandardScaler(), handcrafted_cols),
            ('tfidf', TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True), 'text')
        ])

        # 3. Create DataFrames with preprocessed text
        train_df = X_train[handcrafted_cols].copy()
        train_df['text'] = train_text
        val_df = X_val[handcrafted_cols].copy()
        val_df['text'] = val_text
        test_df = X_test[handcrafted_cols].copy()
        test_df['text'] = test_text

        # 4. Fit transformer on train, transform val and test
        X_train_comb = transformer.fit_transform(train_df)
        X_val_comb = transformer.transform(val_df)
        X_test_comb = transformer.transform(test_df)

        # 5. Class imbalance
        neg = (y_train == 0).sum()
        pos = (y_train == 1).sum()
        scale_pos_weight = neg / pos if pos > 0 else 1.0

        # 6. XGBoost model
        model = XGBClassifier(
            objective='binary:logistic',
            scale_pos_weight=scale_pos_weight,
            n_estimators=2000,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_lambda=1.0,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            early_stopping_rounds=20,
        )

        # Train with early stopping on validation set
        model.fit(
            X_train_comb, y_train,
            eval_set=[(X_val_comb, y_val)],
            verbose=False
        )

        # Evaluate on test set
        y_pred = model.predict(X_test_comb)
        test_f1 = f1_score(y_test, y_pred, average='binary')

        results.append({**config, "test_f1": test_f1})

    return pd.DataFrame(results).sort_values("test_f1", ascending=False).reset_index(drop=True)

In [15]:
handcrafted_cols = [
    'char_len', 'avg_word_length', 'num_hashtags', 'num_mentions',
    'num_urls', 'num_emojis', 'num_exclamations', 'num_replies', 'num_retweets'
]

# results_test = evaluate_preprocessing_configs_on_test(
#    X_train, X_val, X_test,
#    y_train, y_val, y_test,
#    handcrafted_cols=handcrafted_cols
#)
#print(results_test)